Web Scrapping

Lets import import libraries

In [1]:
from bs4 import BeautifulSoup
import requests
import os
import csv
import pandas as pd
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry




Web Scrapper

In [2]:
Base_Url = "https://www.iisermohali.ac.in"
People_Url = "https://www.iisermohali.ac.in/people"
USER_AGENT = "IISERFacultyResearch/1.0"
Timeout = 10 # seconds
TAB_MAP = {
    "Research Area":      "Research Area",
    "Research Interests": "Research Interests",
    "Career Highlights":  "Career Highlights",
    "Awards":             "Awards & Recognitions",
    "Publications":       "Publications",
}

In [3]:
session = requests.Session()
session.headers.update({"USER_AGENT": USER_AGENT})
retry = Retry(
    total = 4,
    backoff_factor = 1,
    status_forcelist = [429, 500, 502, 503, 504],
    allowed_methods=["GET"]  

)
session.mount("https://", HTTPAdapter(max_retries=retry))


In [4]:
def load_robots():
    robots_url = Base_Url + "/robots.txt"

    r = session.get(robots_url, timeout=Timeout)

    rp = RobotFileParser()

    if r.status_code == 200:
        rp.parse(r.text.splitlines())
        print("robots.txt found.")

    elif r.status_code == 404:
        rp.parse([])
        print("No robots.txt found.")

    else:
        raise RuntimeError(
            f"Could not read robots.txt (HTTP {r.status_code})."
        )

    return rp


robots = load_robots()

robots.txt found.


In [5]:
if not robots.can_fetch(USER_AGENT, People_Url):
    raise RuntimeError(
        "robots.txt does not allow scraping the People page."
    )

print("Scraping allowed:", People_Url)

Scraping allowed: https://www.iisermohali.ac.in/people


In [6]:
response = session.get(People_Url, timeout=Timeout)
print("Status Code:", response.status_code)
response.raise_for_status()

Status Code: 200


In [7]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

People


In [8]:
tab_links = soup.select("ul.sppb-nav-custom li a")
tab_panes = soup.find_all("div", class_="sppb-tab-pane")
print("page title:", soup.title.get_text(strip=True))
print("tabs:", len(tab_links), "| tab panes:", len(tab_panes))

for link, pane in zip(tab_links, tab_panes):
    print(f"  {link.get_text(strip=True):<22} tables: {len(pane.find_all('table'))}")
assert len(tab_links) == len(tab_panes), "tab names and tab panes no longer line up: the page layout changed"

page title: People
tabs: 12 | tab panes: 12
  Regular Faculty        tables: 1
  Inspire Faculty        tables: 1
  Lecturers              tables: 1
  Visiting Faculty       tables: 1
  Adjunct Faculty        tables: 1
  Former Faculty         tables: 2
  Honorary Professors    tables: 2
  Emeritus Professors    tables: 1
  Administrative Staff   tables: 1
  Technical Staff        tables: 1
  Institute Fellows      tables: 1
  Project Scientists     tables: 1


In [9]:
SKIP_CATEGORIES = {
    "Administrative Staff",
    "Former Faculty",
    "Technical Staff",
    "Institute Fellows",
    "Project Scientists"
}

In [10]:
def parse_listing(tab_links, tab_panes, skip=SKIP_CATEGORIES):
   
    rows, warnings = [], []
    for link, pane in zip(tab_links, tab_panes):
        category = link.get_text(strip=True)
        if category in skip:
            continue
        for table in pane.find_all("table"):
            trs = table.find_all("tr")
            if not trs:
                continue
            headers = [c.get_text(" ", strip=True) for c in trs[0].find_all(["th", "td"])]
            idx = {h: headers.index(h) for h in ("Name", "Department", "Designation", "Email") if h in headers}
            if "Name" not in idx:
                continue

            for tr in trs[1:]:
                cells = tr.find_all("td")
                if not cells:
                    continue
                if len(cells) != len(headers):
                    warnings.append((category, tr.get_text(" ", strip=True)[:80]))
                if len(cells) <= max(idx.values()):
                    continue                      # row too short to read, listed in warnings above

                def cell(col):
                    return cells[idx[col]].get_text(" ", strip=True) if col in idx else None

                a = cells[idx["Name"]].find("a")
                rows.append({
                    "Name": cell("Name"),
                    "Department": cell("Department"),
                    "Designation": cell("Designation"),
                    "Email": cell("Email"),
                    "Profile Link": a.get("href") if a else None,
                    "Category": category,
                })
    return rows, warnings

def fix_profile_link(x):
    if pd.isna(x):
        return None
    x = str(x)
    if "/faculty/" in x:                         
        x = x[x.find("/faculty/"):]
    return urljoin(Base_Url, x)

listing_rows, listing_warnings = parse_listing(tab_links, tab_panes)
df = pd.DataFrame(listing_rows)
df["Profile Link"] = df["Profile Link"].apply(fix_profile_link)

print("people found:", len(df))
print(df["Category"].value_counts().to_string())
print("rows without a profile link:", df["Profile Link"].isna().sum())

if listing_warnings:
    print(f"\n{len(listing_warnings)} rows whose cell count differs from the header (check these by hand):")
    for cat, txt in listing_warnings:
        print("  ", cat, "|", txt)
df.head()

people found: 135
Category
Regular Faculty        122
Honorary Professors      5
Visiting Faculty         3
Lecturers                2
Emeritus Professors      2
Adjunct Faculty          1
rows without a profile link: 8


,Name,Department,Designation,Email,Profile Link,Category
0,Abhik Ganguli,DMS,Assistant Professor,aganguli,https://www.iisermohali.ac.in/faculty/dms/agan...,Regular Faculty
1,Abhishek Chaudhuri,DPS,Professor,abhishek,https://www.iisermohali.ac.in/faculty/dps/abhi...,Regular Faculty
2,Adrene Freeda Dcruz,HSS,Associate Professor,adrene,https://www.iisermohali.ac.in/faculty/hss/adrene,Regular Faculty
3,Alok Maharana,DMS,Assistant Professor,maharana,https://www.iisermohali.ac.in/faculty/dms/maha...,Regular Faculty
4,Ambresh Shivaji,DPS,Associate Professor,ashivaji,https://www.iisermohali.ac.in/faculty/dps/ashi...,Regular Faculty


In [11]:
LISTING_CSV = "iisermohali_faculty_listing.csv"
df.to_csv(LISTING_CSV, index=False)
print("saved", LISTING_CSV, "->", os.path.abspath(LISTING_CSV))

saved iisermohali_faculty_listing.csv -> e:\SEM 5\IDC409\iisermohali_faculty_listing.csv


In [12]:
def get_section_text(soup, name):
  
    tab = soup.find(
        "a", attrs={"data-toggle": "sppb-tab"},
        string=lambda x: x and x.strip().rstrip(":") == name
    )
    if tab:
        target_id = tab.get("href")
        if target_id:
            content = soup.select_one(target_id)
            if content:
                return content.get_text(" ", strip=True)

    heading = soup.find(
        lambda tag: tag.name in ["h1", "h2", "h3", "h4"]
        and tag.get_text(" ", strip=True).rstrip(":") == name
    )
    if heading:
        parts = []
        for element in heading.find_all_next():
            if element.name in ["h1", "h2", "h3", "h4"] and element != heading:
                break
            text = element.get_text(" ", strip=True)
            if text and text.rstrip(":") != name:
                parts.append(text)
        return " ".join(parts).strip() or None
    return None

In [13]:
def is_own_site(url):
    return isinstance(url, str) and urlparse(url).netloc.endswith("iisermohali.ac.in")

for column in TAB_MAP:
    df[column] = None

def scrape_profile(i, url):
    response = requests.get(url, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for column, website_name in TAB_MAP.items():
        df.at[i, column] = get_section_text(soup, website_name)

todo = [
    (i, u)
    for i, u in df["Profile Link"].items()
    if is_own_site(u)
]

print(f"profiles to visit: {len(todo)}")
failed = []
for n, (i, url) in enumerate(todo, 1):
    try:
        scrape_profile(i, url)
    except Exception as e:
        failed.append((i, url, repr(e)))
    if n % 20 == 0 or n == len(todo):
        print(f"  {n}/{len(todo)} done, {len(failed)} failed so far")

# one retry for failures
still_failed = []
for i, url, _ in failed:
    try:
        scrape_profile(i, url)
    except Exception as e:
        still_failed.append((df.at[i, "Name"], url, repr(e)))
print("\nDONE. Profiles that still failed:", len(still_failed))
for item in still_failed:
    print("  ", item)

profiles to visit: 127
  20/127 done, 0 failed so far
  40/127 done, 0 failed so far
  60/127 done, 0 failed so far
  80/127 done, 0 failed so far
  100/127 done, 0 failed so far
  120/127 done, 0 failed so far
  127/127 done, 1 failed so far

DONE. Profiles that still failed: 1
   ('Dr. T V Venkateswaran', 'https://web.iisermohali.ac.in/dept/hss/tv-venkateswaran/', "HTTPError('403 Client Error: Forbidden for url: https://web.iisermohali.ac.in/dept/hss/tv-venkateswaran/')")


In [14]:
column_order = ["Name", "Department", "Designation", "Email", "Profile Link", "Category"] + list(TAB_MAP)
df = df[column_order]
RAW_CSV = "iisermohali_faculty_raw_final.csv"
df.to_csv(RAW_CSV, index=False)
print("saved", RAW_CSV, "->", os.path.abspath(RAW_CSV), "| shape:", df.shape)

saved iisermohali_faculty_raw_final.csv -> e:\SEM 5\IDC409\iisermohali_faculty_raw_final.csv | shape: (135, 11)


In [15]:
df.describe()

,Name,Department,Designation,Email,Profile Link,Category,Research Area,Research Interests,Career Highlights,Awards,Publications
count,135,135,124,135,127,135,113,114,110,81,114
unique,135,6,5,134,127,6,113,114,110,81,114
top,Abhik Ganguli,DPS,Assistant Professor,-,https://www.iisermohali.ac.in/faculty/dms/agan...,Regular Faculty,Number Theory,"My research interests are in number theory, pa...","Ph.D., University of Chicago",Best Teacher's Award,Abhik Ganguli and Suneel Kumar. On the local c...
freq,1,32,44,2,1,122,1,1,1,1,1
